# Week 43, Day 5: Consolidation, replay and complementary learning systems

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w043/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Learning B destroys A

Write train(W, X, Y, epochs): for each epoch go through the items in random order and apply the delta rule W += eta outer(x, y - x W). Train on A, then on B; separately train from scratch on A and B shuffled together.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(1995)
D, nA, nB = 20, 10, 10
XA, XB = r.normal(size=(nA, D)), r.normal(size=(nB, D))
YA, YB = r.normal(size=(nA, 5)), r.normal(size=(nB, 5))

def err(W, X, Y):
    return np.mean((X @ W - Y) ** 2)

def train(W, X, Y, epochs, eta=0.01, replay=None, frac=0.0):
    W = W.copy()
    for e in range(epochs):
        for i in r.permutation(len(X)):
            ...
            if replay is not None and r.random() < frac:      # used in the next cell
                j = r.integers(len(replay[0]))
                x, y = replay[0][j], replay[1][j]
                W += eta * np.outer(x, y - x @ W)
    return W

W0 = np.zeros((D, 5))
W_A = train(W0, XA, YA, 300)
W_AB = train(W_A, XB, YB, 300)
W_int = train(W0, np.r_[XA, XB], np.r_[YA, YB], 300)
print(f"untrained error on A {np.mean(YA ** 2):.3f}")
print(f"after A: error on A {err(W_A, XA, YA):.4f};  after A then B: A {err(W_AB, XA, YA):.3f}, B {err(W_AB, XB, YB):.4f}")
print(f"interleaved A and B: A {err(W_int, XA, YA):.3f}, B {err(W_int, XB, YB):.3f}")

In [ ]:
import numpy as np
def test_forgets():
    "after learning B, the error on A is above 80% of the untrained baseline"
    assert err(W_AB, XA, YA) > 0.8 * np.mean(YA ** 2)
def test_interleaved():
    "training on A and B together leaves both errors below 0.1"
    assert err(W_int, XA, YA) < 0.1 and err(W_int, XB, YB) < 0.1

test_forgets()
test_interleaved()
print("All checks passed")

### Exercise: How much replay is enough?

Starting from W_A, train on B for 300 epochs with replay=(XA, YA) and frac in 0, 0.1, 0.3 and 1. Store the errors on A and B in res[frac].

In [ ]:
res = {}
for frac in [0.0, 0.1, 0.3, 1.0]:
    W = ...
    res[frac] = (err(W, XA, YA), err(W, XB, YB))
    print(f"replay fraction {frac:.1f}: error on A {res[frac][0]:.3f}, on B {res[frac][1]:.4f}")

In [ ]:
import numpy as np
def test_none():
    "without replay, A is forgotten"
    assert res[0.0][0] > 0.7
def test_some():
    "replay after 10% of B presentations cuts the A error by more than two thirds, and B is still learned"
    assert res[0.1][0] < res[0.0][0] / 3 and res[0.1][1] < 0.05
def test_more():
    "more replay protects A better"
    assert res[1.0][0] < res[0.3][0] < res[0.1][0]

test_none()
test_some()
test_more()
print("All checks passed")

### Exercise: Lesions at different ages

Integrate h and c with Euler steps of 0.1 day from day 0 to day 100, with tau_h = 10 days and k = 0.3 per day. Record c at days 1, 10, 50 and 100, which is what survives a lesion at that age.

In [ ]:
dt_d, tau_h, k = 0.1, 10.0, 0.3
days = np.arange(0, 100 + dt_d / 2, dt_d)
h, c = np.zeros_like(days), np.zeros_like(days)
h[0] = 1.0
for i in range(1, len(days)):
    ...
intact = 1 - (1 - h) * (1 - c)
c_at = {d: c[int(round(d / dt_d))] for d in [1, 10, 50, 100]}
print({d: round(v, 3) for d, v in c_at.items()}, "memory strength surviving a lesion at each age (days)")
plt.plot(days, h, label="hippocampal trace"); plt.plot(days, c, label="neocortical trace (after lesion)"); plt.plot(days, intact, "k:", label="intact")
plt.xlabel("age of memory at lesion (days)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_graded():
    "after a lesion, recent memories are weaker than remote ones"
    assert c_at[1] < c_at[10] < c_at[50]
def test_saturates():
    "remote memories approach full neocortical strength"
    assert c_at[100] > 0.9

test_graded()
test_saturates()
print("All checks passed")